# Notebook 04 — Pipeline Summary & Dataset Validation

**Project:** Starbucks Customer Voice Intelligence: A U.S. Coffee Chain Market Study  
**Purpose:** Confirm the final dataset is clean and correctly typed before any analysis runs. Nothing is transformed here.

## 0. Environment setup

In [1]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
print("Environment ready.")

Environment ready.


## 1. Load analysis-ready dataset

Load `analysis_ready.parquet` from Notebook 03.

In [2]:
df = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 381,999 rows x 33 columns


## 2. Validation assertions

Run assertions on null counts and date range coverage. If anything fails, an error is raised immediately so the problem is visible rather than buried in a downstream chart.

In [3]:
CRITICAL_COLS = [
    "review_id", "review_stars", "date", "text",
    "brand_category", "sentiment_label", "star_tier",
]

print("=== Null assertions ===")
for col in CRITICAL_COLS:
    null_count = df[col].isnull().sum()
    assert null_count == 0, f"FAIL: {col} has {null_count} nulls"
    print(f"  OK  {col}")

assert df["date"].min() >= pd.Timestamp("2017-01-01")
assert df["date"].max() <= pd.Timestamp("2021-12-31")
print(f"\n  OK  date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print("\nAll assertions passed.")

=== Null assertions ===
  OK  review_id
  OK  review_stars
  OK  date
  OK  text
  OK  brand_category
  OK  sentiment_label
  OK  star_tier

  OK  date range: 2017-01-01 -> 2021-12-31

All assertions passed.


## 3. Data dictionary

Field-by-field reference for `analysis_ready.parquet`: type, description, and an example value.

In [ ]:
DATA_DICT = [
    ("review_id",             "Unique review identifier"),
    ("user_id",               "Unique reviewer identifier"),
    ("business_id",           "Unique business identifier"),
    ("review_stars",          "Star rating given by reviewer (1-5)"),
    ("date",                  "Review submission date"),
    ("text",                  "Raw review text"),
    ("useful",                "Yelp 'useful' votes received"),
    ("funny",                 "Yelp 'funny' votes received"),
    ("cool",                  "Yelp 'cool' votes received"),
    ("name",                  "Business name"),
    ("city",                  "City (raw)"),
    ("state",                 "U.S. state abbreviation"),
    ("business_avg_stars",    "Business lifetime avg star rating on Yelp"),
    ("business_review_count", "Business lifetime review count on Yelp"),
    ("user_name",             "Reviewer display name"),
    ("user_review_count",     "Reviewer lifetime review count"),
    ("yelping_since",         "Date reviewer joined Yelp"),
    ("user_avg_stars",        "Reviewer lifetime avg star rating"),
    ("star_tier",             "Satisfaction tier: Critical(1-2) / Neutral(3) / Positive(4-5)"),
    ("brand_category",        "Competitive segment: Starbucks / Dunkin' / Dutch Bros / Independent Café / ..."),
    ("review_length",         "Review word count"),
    ("length_tier",           "Short(<50w) / Medium(50-150w) / Long(>150w)"),
    ("user_activity_tier",    "Reviewer tier: Casual / Regular / Power / Elite"),
    ("topic_tags",            "Pipe-separated multi-label topics, e.g. 'Service|Food Quality'. Filter with str.contains()."),
    ("year",                  "Review year"),
    ("month",                 "Review month (1-12)"),
    ("quarter",               "Review quarter (1-4)"),
    ("day_of_week",           "Day of week (0=Monday, 6=Sunday)"),
    ("year_quarter",          "Year-quarter label, e.g. 2019Q3"),
    ("is_weekend",            "True if review posted on Saturday or Sunday"),
    ("sentiment_score",       "VADER compound score [-1.0, 1.0]"),
    ("sentiment_label",       "Positive / Neutral / Negative"),
]

print(f"{'Column':<25} {'Type':<15} {'Example':<35} Description")
print("-" * 110)
for col, desc in DATA_DICT:
    if col in df.columns:
        dtype = str(df[col].dtype)
        example = str(df[col].iloc[0])[:32]
        print(f"{col:<25} {dtype:<15} {example:<35} {desc}")

## 4. Pipeline summary

What the pipeline scanned, filtered, and retained end to end. These figures are cited in the project README and executive summary.

In [5]:
sbux   = df[df["brand_category"] == "Starbucks"]
market = df[df["brand_category"] != "Starbucks"]

print("=" * 58)
print("PIPELINE SUMMARY")
print("=" * 58)
print(f"  Time window           : 2017-01-01 -> 2021-12-31")
print(f"  Total reviews         : {len(df):,}")
print(f"  Starbucks reviews     : {len(sbux):,}")
print(f"  Market benchmark      : {len(market):,}")
print(f"  Unique businesses     : {df['business_id'].nunique():,}")
print(f"  States covered        : {df['state'].nunique()}")
print(f"  Cities covered        : {df['city'].nunique():,}")
print(f"  Total columns         : {df.shape[1]}")
print("=" * 58)
print(f"  Starbucks avg rating          : {sbux['review_stars'].mean():.2f}")
print(f"  Market avg rating             : {market['review_stars'].mean():.2f}")
print(f"  Starbucks % Positive sentiment: {(sbux['sentiment_label']=='Positive').mean()*100:.1f}%")
print(f"  Market % Positive sentiment   : {(market['sentiment_label']=='Positive').mean()*100:.1f}%")
print("=" * 58)
print(f"  Starbucks star tier breakdown:")
for tier, count in sbux["star_tier"].value_counts().items():
    pct = count / len(sbux) * 100
    print(f"    {tier:<10} {count:>6,}  ({pct:.1f}%)")
print("=" * 58)

PIPELINE SUMMARY
  Time window           : 2017-01-01 -> 2021-12-31
  Total reviews         : 381,999
  Starbucks reviews     : 11,675
  Market benchmark      : 370,324
  Unique businesses     : 6,909
  States covered        : 13
  Cities covered        : 508
  Total columns         : 33
  Starbucks avg rating          : 2.90
  Market avg rating             : 3.97
  Starbucks % Positive sentiment: 63.1%


  Market % Positive sentiment   : 87.4%


  Starbucks star tier breakdown:
    Critical    5,557  (47.6%)
    Positive    4,947  (42.4%)
    Neutral     1,171  (10.0%)


---

**Next: Notebook 05 — Review Volume Trends**